# 10j The fMRI headroom fit on split halves (NSD)

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 10a and 10b.

Development analysis, not a prespecified test. The headroom fit in the paper measures gain and borrowing on the same test images. Here borrowing's accuracy
comes from one random half of the test images and the map's gain from the other (five splits, both directions),
against NSD's noise ceiling.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '10j-headroomsplit', 'cpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"K": 200, "draws": [13, 14, 15], "folds": 5, "n_galleries": 4, "gallery_size": 200, "gallery_seed": 346, "delta_r_share": 0.05, "roi_classes": ["prf-visualrois", "floc-faces", "floc-places", "floc-bodies", "floc-words"], "ks": [250, 500], "half_seed": 402, "n_halves": 5}')
SUBJECTS = ['subj01', 'subj02', 'subj03', 'subj04', 'subj05', 'subj06', 'subj07', 'subj08']

import numpy as np
import pandas as pd
from cross_subject_encoding.eval import lowrank_map as lm
from cross_subject_encoding.eval import nsd_confirm as nc
from cross_subject_encoding.models.ridge import RidgePerOutput
from cross_subject_encoding.utils import config as C

if not isinstance(FIXED["K"], int):
    raise RuntimeError("K is not fixed: fill it in from 09q before the confirmation runs")
IN = Path("/kaggle/input")
conv = next(p.parent for p in IN.rglob("shared_train_nsd_ids.npy"))
shared = np.load(conv / "shared_train_nsd_ids.npy")
cols, common = {}, {}
for h in ("lh", "rh"):
    masks = [np.load(next((conv / f"{s}_roi").rglob(f"{h}.all-vertices_fsaverage_space.npy"))) for s in SUBJECTS]
    common[h], cl = nc.common_vertices(masks)
    cols[h] = dict(zip(SUBJECTS, cl))
V = sum(len(common[h]) for h in common)
ids, Y = {}, {}
for s in SUBJECTS:
    ids[s] = np.load(conv / f"{s}_train_nsd_ids.npy")
    Y[s] = np.concatenate([np.load(conv / f"{s}_{h}_fmri.npy", mmap_mode="r")[:, cols[h][s]] for h in ("lh", "rh")], 1)
noise = np.concatenate([np.concatenate([np.load(conv / f"{s}_{h}_noise_ceiling.npy").astype(np.float64)[cols[h][s]] for h in ("lh", "rh")]) for s in SUBJECTS])
DELTA_R = FIXED["delta_r_share"] * float(np.median(np.sqrt(np.clip(noise, 0, None) / 100)))
FEATS = {}
for name in ("alexnet", "dinov2"):
    f = np.load(next(IN.rglob(f"10b_{name}_kpca1000.npz")))
    FEATS[name] = (f["z"].astype(np.float64), {int(i): j for j, i in enumerate(f["nsd_ids"])})
feats = lambda name, nsd_ids: FEATS[name][0][[FEATS[name][1][int(i)] for i in nsd_ids]]
test_ids = np.sort(shared)
rows_of = {s: {int(i): j for j, i in enumerate(ids[s])} for s in SUBJECTS}
test_rows = {s: np.array([rows_of[s][int(i)] for i in test_ids]) for s in SUBJECTS}
pool_rows = {s: np.setdiff1d(np.arange(len(ids[s])), test_rows[s]) for s in SUBJECTS}
GALS = nc.galleries(len(test_ids), FIXED["n_galleries"], FIXED["gallery_size"], FIXED["gallery_seed"])
rois = {}
for cls in FIXED["roi_classes"]:
    for s in SUBJECTS:
        lab = np.concatenate([np.load(next((conv / f"{s}_roi").rglob(f"{h}.{cls}_challenge_space.npy")))[cols[h][s]] for h in ("lh", "rh")])
        rois[(s, cls)] = lab
print(f"{V} common vertices; delta_r {DELTA_R:.4f}; {len(test_ids)} test images; pools", {s: len(pool_rows[s]) for s in SUBJECTS},
      "|", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
def ols_coef(x, y):
    xa = np.concatenate([np.ones((len(x), 1)), x], 1)
    return np.linalg.solve(xa.T @ xa, xa.T @ np.asarray(y, np.float64)).astype(np.float32)
ols_pred = lambda beta, x: (np.concatenate([np.ones((len(x), 1)), x], 1) @ beta.astype(np.float64))
BETA = {s: ols_coef(feats("alexnet", ids[s][pool_rows[s]]), Y[s][pool_rows[s]]) for s in SUBJECTS}
print("alexnet encoders fitted |", f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)

In [ ]:
def r_vert(p, m):
    p, m = p - p.mean(0), m - m.mean(0)
    return (p * m).sum(0) / np.sqrt((p ** 2).sum(0) * (m ** 2).sum(0) + 1e-30)

def fit(gain, ceil, borrow):
    x = np.column_stack([np.ones_like(ceil), ceil, borrow])
    coef, *_ = np.linalg.lstsq(x, gain, rcond=None)
    r2 = 1 - ((gain - x @ coef) ** 2).sum() / ((gain - gain.mean()) ** 2).sum()
    return {"intercept": float(coef[0]), "ceiling": float(coef[1]), "borrowing": float(coef[2]), "r2": float(r2)}

N = len(test_ids)
rows, pooled = [], {"full": [], "split": [], "same_half": []}
for p, s in enumerate(SUBJECTS, start=1):
    t = time.monotonic()
    donors = [d for d in SUBJECTS if d != s]
    meas = np.asarray(Y[s][test_rows[s]], np.float64)
    ceil = np.sqrt(np.clip(np.concatenate([np.load(conv / f"{s}_{h}_noise_ceiling.npy").astype(np.float64)[cols[h][s]] for h in ("lh", "rh")]), 0, None) / 100)
    y_pool = np.asarray(Y[s][pool_rows[s]], np.float32)
    x_pool, x_test = feats("alexnet", ids[s][pool_rows[s]]), feats("alexnet", test_ids)
    def donor_mean(x):
        acc = np.zeros((len(x), V), np.float32)
        for d in donors:
            acc += ols_pred(BETA[d], x).astype(np.float32)
        return acc / len(donors)
    a_pool, a_test = donor_mean(x_pool), donor_mean(x_test).astype(np.float64)
    mean, B = lm.prediction_basis(a_pool, FIXED["K"])
    maps = []
    for d in FIXED["draws"]:
        order = np.random.default_rng([d, p]).permutation(len(pool_rows[s]))
        for k in FIXED["ks"]:
            idx = order[:k]
            maps.append(lm.apply_lowrank_map(lm.fit_lowrank_map(a_pool[idx].astype(np.float64), y_pool[idx].astype(np.float64), mean, B, seed=d), a_test, mean, B))
    m_test = np.mean(maps, 0)
    del maps, a_pool
    # full test set, as in the paper's headroom fit (the map averaged over draws and budgets here, so slightly different numbers)
    g_full, b_full = r_vert(m_test, meas) - r_vert(a_test, meas), r_vert(a_test, meas)
    pooled["full"].append((g_full, ceil, b_full))
    rows.append({"subject": s, "kind": "full", **fit(g_full, ceil, b_full)})
    # split halves: borrowing's r from one half, the gain from the other, both directions, several random splits
    # same-half fits (gain and borrowing from the same half) have the same noise level as the split ones but share
    # the noise: the split minus same-half difference isolates the shared-noise part; split vs full also reflects
    # the noisier half-size estimates (attenuation)
    gs, bs, bq = [], [], []
    for j in range(FIXED["n_halves"]):
        h = np.random.default_rng([FIXED["half_seed"], p, j]).permutation(N)
        A, Bh = h[:N // 2], h[N // 2:]
        for P, Q in ((A, Bh), (Bh, A)):
            gs.append(r_vert(m_test[Q], meas[Q]) - r_vert(a_test[Q], meas[Q]))
            bs.append(r_vert(a_test[P], meas[P]))
            bq.append(r_vert(a_test[Q], meas[Q]))
            rows.append({"subject": s, "kind": f"split{j}", **fit(gs[-1], ceil, bs[-1])})
            rows.append({"subject": s, "kind": f"same{j}", **fit(gs[-1], ceil, bq[-1])})
    pooled["split"].append((np.concatenate(gs), np.tile(ceil, len(gs)), np.concatenate(bs)))
    pooled["same_half"].append((np.concatenate(gs), np.tile(ceil, len(gs)), np.concatenate(bq)))
    print(s, {r["kind"]: (round(r["ceiling"], 3), round(r["borrowing"], 3), round(r["r2"], 3)) for r in rows[-5:]}, f"{time.monotonic() - t:.0f} s |",
          f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)

In [ ]:
frame = pd.DataFrame(rows)
frame.to_csv(C.METRICS_DIR / "10j_fits.csv", index=False)
res = {"post_hoc": "development: NSD people of P-7; split-half check of the headroom fit", "per_person": {}, "pooled": {}}
for s in SUBJECTS:
    f = frame[frame.subject == s]
    sp, sm = f[f.kind.str.startswith("split")], f[f.kind.str.startswith("same")]
    res["per_person"][s] = {"full": f[f.kind == "full"].iloc[0][["ceiling", "borrowing", "r2"]].astype(float).to_dict(),
                            "split_mean": sp[["ceiling", "borrowing", "r2"]].mean().astype(float).to_dict(),
                            "same_half_mean": sm[["ceiling", "borrowing", "r2"]].mean().astype(float).to_dict()}
for kind, parts in pooled.items():
    g, c, b = (np.concatenate([q[i] for q in parts]) for i in range(3))
    res["pooled"][kind] = fit(g, c, b)
# the test: how much of the same-half borrowing loading survives when the noise is no longer shared
res["borrowing_loading_kept_vs_same_half"] = res["pooled"]["split"]["borrowing"] / res["pooled"]["same_half"]["borrowing"]
res["borrowing_loading_kept_vs_full"] = res["pooled"]["split"]["borrowing"] / res["pooled"]["full"]["borrowing"]
res["people_split_borrowing_negative"] = int(sum(v["split_mean"]["borrowing"] < 0 for v in res["per_person"].values()))
(C.METRICS_DIR / "10j_summary.json").write_text(json.dumps(res, indent=2))
print("10J_SUMMARY_JSON " + json.dumps(res))